In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm


%matplotlib inline
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Dataset shape: {df_food.shape}")


In [ ]:
# Task 2: Write your code here:
df_food.head()

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
# 1. What does our target variable (charges) look like?

plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delaviry Distribution')
plt.xlabel('Distance_km')
plt.ylabel('Preparation_Time_min')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df_food.drop(columns=['Order_ID']).copy()

df_clean


In [ ]:
# Task 2: Write your code here:
# 1. Calculate the percentage of missing values for each column
# .isnull() checks for missing values, .sum() counts them, and we divide by total rows (len)
missing_percentage = (df_clean.isnull().sum() / len(df_clean)) * 100

# 2. Create a new DataFrame to store the results cleanly
# We map the column names (index) to their calculated missing percentages
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
# 3. Filter and Sort the results
# Filter: Keep only columns where Missing_Percentage is greater than 0
# Sort: Order the results from highest missing % to lowest (ascending=False)
missing_data = missing_data[missing_data['Missing_Percentage'] > 0] \
    .sort_values('Missing_Percentage', ascending=False)

# 4. Display the results
print("Missing Data Analysis:")
# Show the top 10 columns with the most missing data
missing_data.head(10)



In [ ]:
# 2. Define a function to hunt for Missing Data (NaNs)
# Missing values are "holes" in your dataset.
# Most Machine Learning models (like Logistic Regression) will crash immediately
# if you feed them a blank space/NaN.
def check_missing_values(df):
    # 1. Calculate the sum of missing cells for every column
    # .isnull(): Returns a True/False table (True if missing).
    # .sum(): Adds up the "True" values (since True = 1) to get the total count.
    missing_values = df.isnull().sum()

    # 2. Print ONLY the columns that have problems
    # missing_values > 0: This filter hides the "clean" columns (where count is 0).
    # This keeps your output readable so you don't scroll through 50 columns of "0".
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])

    # 3. Global Check
    # .any(): Returns True if there is even a single missing value anywhere in the list.
    if missing_values.sum() > 0:
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

# 4. Run the function
check_missing_values(df_clean)

In [ ]:
# YOUR CODE HERE
print(f"Delivery_Time: {df_clean['Delivery_Time'].sum()}")
print(f"Delivery_Time normal: {(df_clean['Delivery_Time'] == 0).sum()}")

In [ ]:
# Do we have categorical columns?
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']:
    df_clean[col] = df_clean[col].fillna('unknown')


# Fill cylinders with mode - discrete feature, mode is most representative
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])



In [ ]:
df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean(), inplace=True)

In [ ]:
print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
# 1. Imports the OneHotEncoder class from sklearn's preprocessing module.
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder



cati = [ ['Weather'], ['Traffic_Level'], ['Time_of_Day'], ['Vehicle_Type']]

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(cati)

print("Original:", cati)
print("Encoded:\n", encoded)



In [ ]:
df_clean.columns


In [ ]:
# numerical_cols = df_clean.select_dtypes(include=["number"]).columns
# print("Numerical:", list(numerical_cols))

In [ ]:
# 1. Import Cross-Validation Tool
# KFold: A technique to split data into 'K' different parts (folds).
# Instead of training once on one train/test split, you train K times (e.g., 5 times).
# Each time, a different part is used as the "Test" set.
# This gives a much more reliable estimate of model performance.
from sklearn.model_selection import KFold

# 2. Import Evaluation Metrics
# sklearn_mse: Calculates Mean Squared Error. (Average of squared differences).
# Used to penalize large errors heavily. The lower, the better.
from sklearn.metrics import mean_squared_error as sklearn_mse

# mean_absolute_error (MAE): Average of absolute differences.
# It's easier to interpret than MSE because it's in the same units as your target (e.g., "Dollars").
# Formula: mean(|y_true - y_pred|)
from sklearn.metrics import mean_absolute_error

# r2_score (R-Squared): A score between 0 and 1 (or negative if the model is terrible).
# It tells you "How much of the variance in the data does my model explain?"
# 1.0 = Perfect fit. 0.0 = As good as just guessing the average.
from sklearn.metrics import r2_score

In [ ]:
from sklearn.preprocessing import StandardScaler

# Pick only the numerical columns, NOT the target

scaler = StandardScaler()



df_clean[encoded] = scaler.fit_transform(df_clean[encoded])

# 4. Verify
df_clean.head()



In [ ]:
# Mean Squared Error (MSE) - Cost Function Implementation
def mean_squared_error(y, y_hat):
    # 1. Calculate the difference (Error)
    # (y_hat - y): Vectorized subtraction. Calculates the error for every single data point at once.

    # 2. Square the errors
    # ** 2: We square the errors so that negative errors don't cancel out positive ones.
    # It also penalizes large errors more heavily than small ones.

    # 3. Sum the squared errors
    # np.sum(...): Adds up the squared errors for the entire dataset.

    # 4. Calculate the Average (with a twist)
    # len(y): The number of training examples (often denoted as 'm').
    # 1 / (2 * len(y)): This is the specific "Cost Function" version of MSE.
    # The standard statistical MSE is just 1/m.
    # The extra '2' is added to make the derivative (Gradient Descent) calculation cleaner later
    # (because the derivative of x^2 is 2x, the 2s cancel out).
    return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)

In [ ]:
def gradient_descent(X, y, learning_rate, n_iters=500):
    # 1. Get dataset dimensions
    # m = Number of training examples (rows)
    # n = Number of features/inputs (columns)
    m, n = X.shape

    # 2. Initialize Weights (Theta)
    # We start with all weights at 0.0. The model currently "knows" nothing.
    # If we have 3 features, theta will be [0, 0, 0].
    theta = np.zeros(n)
    losses = []

    # 3. Start the Training Loop
    # tqdm is just a library to show a progress bar so you know how fast it's going.
    for _ in tqdm(range(n_iters), desc="Training Linear Regression"):

        # A. Make Predictions (Hypothesis)
        # We multiply our Input (X) by our current Weights (theta).
        # Math: h(x) = w * x
        y_hat = np.dot(X, theta)

        # B. Calculate the Gradient (The Direction)
        # This formula calculates the derivative of the Cost Function.
        # It tells us: "In which direction should we move the weights to reduce error?"
        # (y_hat - y): The error (prediction - actual).
        # np.dot(X.T, ...): Multiplies errors by input values to find the slope.
        # / m: Averages the gradient across all examples.
        gradient = np.dot(X.T, (y_hat - y)) / m

        # C. Update the Weights (The Step)
        # We move 'theta' in the opposite direction of the gradient.
        # learning_rate: Controls how big the step is. Too big = overshoot; Too small = slow.
        theta -= learning_rate * gradient

        # D. Track Progress
        # We calculate the current error to see if the model is actually learning.
        loss = mean_squared_error(y, y_hat)
        losses.append(loss)

    # 4. Return the learned weights and the history of errors
    return theta, losses

In [ ]:
n_splits = 5  # K=5 Folds

# Initialize the Cross-Validation Splitter
# n_splits=5: The data will be cut into 5 equal pieces (folds).
# shuffle=True: CRITICAL. This randomizes the order of the rows before splitting.
#    Without this, if your data is sorted (e.g., all "Type A" first, then "Type B"),
#    one fold might get ONLY Type A and another ONLY Type B, ruining the training.
# random_state=42: Ensures the "random" shuffle is exactly the same every time you run the code.
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# Initialize empty lists to store the performance scores for each of the 5 folds.
# Since K-Fold runs 5 times, we need these containers to save the results of each run
# so we can calculate the average performance at the end.

# Stores the value of the Cost Function (J) if you are tracking it manually
lr_losses = []

# Stores Mean Squared Error (Average squared difference between predicted and actual)
lr_mse = []

# Stores Root Mean Squared Error (Square root of MSE)
# This is often preferred because it returns the error to the original units (e.g., "Dollars").
lr_rmse = []

# Stores R-Squared Score (Coefficient of determination)
# 1.0 is perfect, 0.0 is terrible. This tells us how well the model fits the data variance.
lr_r2 = []

In [ ]:
# Loop through each of the 5 folds
# kf.split(X) returns two lists of row numbers:
# 1. train_index: Roughly 80% of the rows to learn from.
# 2. test_index: The remaining 20% of rows to test on.
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    # 1. Slice the data using the indices
    # We use .iloc[] because train_index gives us integer positions (e.g., row 0, row 5, row 10).
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 2. Train the Model
    # We call the custom gradient_descent function we defined earlier.
    # It returns 'theta' (the learned weights/coefficients) and 'losses' (history of error reduction).
    # .values converts the Pandas DataFrame to a NumPy array for the math calculations.
    theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

    # 3. Validate (Make Predictions)
    # We apply the learned weights (theta) to the test data.
    # Math: y_pred = X_test * theta
    y_pred = np.dot(X_test.values, theta)

    # 4. Calculate Evaluation Metrics
    # MSE: Standard error metric.
    mse = sklearn_mse(y_test, y_pred)
    # RMSE: Square root of MSE. Use this to see error in "Real Units" (e.g., Sales Dollars).
    rmse = np.sqrt(mse)
    # R2 Score: Accuracy percentage (0.0 to 1.0). How well did the line fit?
    r2 = r2_score(y_test, y_pred)

    # 5. Store Results
    # Save the scores for this specific fold so we can average them later.
    lr_losses.append(losses)
    lr_mse.append(mse)
    lr_rmse.append(rmse)
    lr_r2.append(r2)

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler


# Define features (X) and target (y)
# X: The input data the model uses to learn
feature_cols =['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df_clean[feature_cols]

# y: The answer we want to predict
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
# random_state=42: ensures reproducible results (fixed seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2,3,4,5: Write your code here:

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: